In [1]:
import pandas as pd
import numpy as np

# aggregated_results is produced by gencsv.py, which read all SUMMARY_* files and parse them into one CSV
# now contains 10 reps data for all 6 subjects, with ENHANCED/REDUCED results for FIX scenario, and deepseek-r1 results
data_path = "aggregated_results.csv"
data = pd.read_csv(data_path)
data.head()

,file,scenario,git_info,max_iter,LLM,LLM_temp,NOFEEDBACK,GENCMD,RUNFUZZ,subject,commit,final_result,unintended_bug,time,iid,score,success
0,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,b7e3bae,R,False,100,5013,1,0
1,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,70e275e,D,False,97,5138,2,0
2,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,53b61c1,D,False,57,5117,2,0
3,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,70e275e,B,False,36,5153,3,1
4,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,b7e3bae,R,False,107,5013,1,0


In [2]:
# check the unique values in the column
for column in data.columns:
    if column in ["file", "time"]:
        continue
    print(f"{column}: {data[column].unique()}")

# get number of total iids
N_IID = data['iid'].nunique()
N_DATAPOINT = N_IID * 20  # repeat 10 times  * 2 scenarios for each iid per setting
print(f"Total number of bug issues (iids): {N_IID}")
print(f"Each setting should have {N_DATAPOINT} data points")

scenario: <StringArray>
['BIC', 'FIX']
Length: 2, dtype: str
git_info: <StringArray>
['DIFFONLY', 'FULL', 'MSGONLY', 'ENHANCED', 'REDUCED']
Length: 5, dtype: str
max_iter: [ 5 10]
LLM: <StringArray>
['gpt-4o-2024-08-06', 'deepseek-r1', 'gpt-4o-mini']
Length: 3, dtype: str
LLM_temp: [0.5 1. ]
NOFEEDBACK: [nan  1.]
GENCMD: [nan  1.]
RUNFUZZ: [nan]
subject: <StringArray>
['jerryscript',          'jq',     'libxml2', 'micropython',        'mujs',
     'php-src',     'poppler',          'z3']
Length: 8, dtype: str
commit: <StringArray>
['b7e3bae', '70e275e', '53b61c1', '3686410', '95cc5e9', 'd7e2125', 'de51531',
 '680baef', '0e067ef', '4003202', '3fa10e8', '71c2ab5', 'de21386', '499c91b',
 '9a82b94', '74c84a8', '1406b20', '14604a4', '9cfc723', 'd0c3f01', '6273df6',
 'db21cd5', '3ad7f81', 'dc6270d', '4c93955', '1fef566', '5e9189d', '69ead7d',
 'e702046', '7e14680', 'f397a3e', '908ab1c', '3185bb5', 'c0252d7', '4b013ec',
 '0615d13', '8c27b12', '832e069', '4c7f6be', '3f71a1c', '833f82c', '6871e

In [3]:
# get results for a specific config, like MSGONLY
data_msgonly = data[data['git_info'] == 'MSGONLY']
# count success of msgonly
data_msgonly_success = data_msgonly[data_msgonly['success'] == 1]
# unique by commit
data_msgonly_success_unique = data_msgonly_success.drop_duplicates(subset=['subject', 'commit'])
# group by scenario
data_msgonly_success_unique = data_msgonly_success_unique.groupby(['scenario']).size().reset_index(name='count')
display(data_msgonly_success_unique)

# get results where final_result is 'X', need to manually inspect reachability and score
data_x = data[(data['final_result'] == 'X') & (data['git_info'] == 'FULL') & (data['max_iter'] == 5) & (data['LLM'] == 'gpt-4o-2024-08-06')]
# unique on iid and scenario, sorted by iid
data_x = data_x.drop_duplicates(subset=['subject', 'commit', 'iid'])
data_x = data_x.sort_values(by=['iid'])
print(data_x.to_csv())

,scenario,count
0,BIC,7
1,FIX,13


,file,scenario,git_info,max_iter,LLM,LLM_temp,NOFEEDBACK,GENCMD,RUNFUZZ,subject,commit,final_result,unintended_bug,time,iid,score,success
3421,SUMMARY_mujs_BIC_GITFULL_ITER5_gpt-4o-2024-08-06_TEMP0.5_250905-2232.txt,BIC,FULL,5,gpt-4o-2024-08-06,0.5,,,,mujs,832e069,X,True,46,141,2,0
2091,SUMMARY_libxml2_FIX_GITFULL_ITER5_gpt-4o-2024-08-06_TEMP0.5_GENCMD_260203-0916.txt,FIX,FULL,5,gpt-4o-2024-08-06,0.5,,1.0,,libxml2,6273df6,X,True,44,550,0,0
4891,SUMMARY_poppler_BIC_GITFULL_ITER5_gpt-4o-2024-08-06_TEMP0.5_250827-1940.txt,BIC,FULL,5,gpt-4o-2024-08-06,0.5,,,,poppler,3cae777,X,True,126,1289,0,0
128,SUMMARY_jerryscript_BIC_GITFULL_ITER5_gpt-4o-2024-08-06_TEMP0.5_250826-2145.txt,BIC,FULL,5,gpt-4o-2024-08-06,0.5,,,,jerryscript,b7e3bae,X,True,78,5013,1,0
1219,SUMMARY_jq_FIX_GITFULL_ITER5_gpt-4o-2024-08-06_TEMP0.5_260117-0915.txt,FIX,FULL,5,gpt-4o-2024-08-06,0.5,,,,jq,499c91b,X,True,47,49014,0,0
867,SUMMARY_jq_BIC_GITFULL_ITER5_gpt-4o-2024-08-06_TEMP0.5_260117-0919.txt,BIC,FULL,5,gpt-4o-2024-08-

In [4]:
try:
    subject_order = pd.read_csv("targets.csv")["software"].unique().tolist()
except:
    subject_order = ["mujs", "libxml2", "poppler", "jerryscript", "z3", "php-src", "jq", "micropython"]
print(f"Subject order: {subject_order}")

data["subject"] = pd.Categorical(data["subject"], categories=subject_order, ordered=True)

Subject order: ['mujs', 'libxml2', 'poppler', 'jerryscript', 'z3', 'php-src', 'jq', 'micropython']


### Table 2
Results for bug finding and bug reproduction effectiveness of Cleverest (default configuration)

In [5]:
# config for default
config_def = {
    "git_info": "FULL",
    "max_iter": 5,
    "LLM": "gpt-4o-2024-08-06",
    "LLM_temp": 0.5,
    "NOFEEDBACK": np.nan,  # choose rows with NaN
    "GENCMD": np.nan,
    # "RUNFUZZ": np.nan,
}
# subset the data regarding the confiG
data_def = data.copy()
for key, value in config_def.items():
    if pd.isna(value):
        data_def = data_def[pd.isna(data_def[key])]
    else:
        data_def = data_def[data_def[key] == value]
# check the number of datapoints; expected to be 720 = 36 iid * 2 scenario * 10 repetition; if not, print warning
if len(data_def) != N_DATAPOINT:
    print(
        f"!!Warning!!: the number of datapoints is {len(data_def)}; expected to be {N_DATAPOINT}"
    )
data_def["reached"] = data_def["score"].apply(lambda x: 1 if x >= 1 else 0)
data_def["changed"] = data_def["score"].apply(lambda x: 1 if x >= 2 else 0)
# aggregate the data regarding the scenario and iid
data_def = (
    data_def.groupby(["scenario", "subject", "iid"])
    # .agg({"score": "mean", "success": "sum", "time": "mean"})
    .agg({"score": "mean", "reached": "sum", "changed": "sum", "success": "sum", "time": "mean"})
    .reset_index()
)
# scenario to multi columns, scenario first then (score, success, time)
data_def_pivot = data_def.pivot(index=["subject", "iid"], columns="scenario")
# order the column index by Subject (level 0) first, then IID (level 1)
data_def_pivot = data_def_pivot.sort_index(level=[0, 1])
# add average row
average = data_def_pivot.agg({
    ("score", "BIC"): "mean",
    ("score", "FIX"): "mean",
    ("time", "BIC"): "mean",
    ("time", "FIX"): "mean",
    ("reached", "BIC"): lambda x: (x != 0).sum(),  # Count non-zero values
    ("reached", "FIX"): lambda x: (x != 0).sum(),
    ("changed", "BIC"): lambda x: (x != 0).sum(),  # Count non-zero values
    ("changed", "FIX"): lambda x: (x != 0).sum(),
    ("success", "BIC"): lambda x: (x != 0).sum(),  # Count non-zero values
    ("success", "FIX"): lambda x: (x != 0).sum(),
}).to_frame().T
average.index = pd.MultiIndex.from_tuples(
    [(r"\multicolumn{2}{c|}{Average}", "")]
)
data_def_pivot = pd.concat([data_def_pivot, average])
# check the column types
# display(data_def_pivot)
# sort data_def by scenario, and then iid
# Define subject order from targets.csv to ensure correct grouping (based on targets.csv order)
try:
    subject_order = pd.read_csv("targets.csv")["software"].unique().tolist()
except:
    subject_order = ["mujs", "libxml2", "poppler", "jerryscript", "z3", "php-src", "jq", "micropython"]

data_def["subject"] = pd.Categorical(data_def["subject"], categories=subject_order, ordered=True)

# sort data_def by subject, iid, and scenario
data_def = data_def.sort_values(by=["subject", "iid", "scenario"])
display(data_def)

,scenario,subject,iid,score,reached,changed,success,time
0,BIC,mujs,65,3.0,10,10,10,7.5
36,FIX,mujs,65,2.2,10,6,6,19.1
1,BIC,mujs,141,2.0,10,10,0,45.4
37,FIX,mujs,141,2.1,10,10,1,31.7
2,BIC,mujs,145,2.6,10,8,8,37.1
...,...,...,...,...,...,...,...,...
69,FIX,micropython,17815,3.0,10,10,10,5.5
34,BIC,micropython,17841,2.0,10,10,0,21.4
70,FIX,micropython,17841,1.0,10,0,0,39.2
35,BIC,micropython,17847,2.0,10,10,0,40.9


Formatting for the latex table

In [6]:
# order: iid
data_def_pivot_latex = data_def_pivot.copy()

###### index formats
# mujs -> \mujs
# libxml2 -> \libxml
# poppler -> \poppler
# jerryscript -> \jerryscript
# php-src -> \php
# z3 -> \zthree
##### also the level 1 should be "3" -> "\#3"
data_def_pivot_latex.index = data_def_pivot_latex.index.set_levels(
    [
        data_def_pivot_latex.index.levels[0].map(
            {
                "mujs": r"\mujs",
                "libxml2": r"\libxml",
                "poppler": r"\poppler",
                "jerryscript": r"\jerryscript",
                "php-src": r"\php",
                "z3": r"\zthree",
                "micropython": r"\micropython",
                "jq": r"\jq",
                r"\multicolumn{2}{c|}{Average}": r"\multicolumn{2}{c|}{Average}",
            }
        ),
        data_def_pivot_latex.index.levels[1].map(lambda x: f"\\#{x}" if x else ""),
    ]
)

###### column formats
# score -> \SLIDER{3.3em}{score / 3:.2f}{circle}
data_def_pivot_latex["score"] = data_def_pivot_latex["score"].map(
    lambda x: f"\\SLIDER{{3.3em}}{{{(x/3):.2f}}}{{circle}}"
)
data_def_pivot_latex["reached"] = data_def_pivot_latex["reached"].map(
    lambda x: f"{int(x)}/10"
)
data_def_pivot_latex["changed"] = data_def_pivot_latex["changed"].map(
    lambda x: f"{int(x)}/10"
)
# success -> {success}/10
data_def_pivot_latex["success"] = data_def_pivot_latex["success"].map(
    lambda x: f"{int(x)}/10"
)
# time -> {time:.1f}
data_def_pivot_latex["time"] = data_def_pivot_latex["time"].map(
    lambda x: f"{x:.1f}"
)

# Fix Average row denominator: X/10 -> X/{N_IID}
avg_idx = (r"\multicolumn{2}{c|}{Average}", "")
for col in ["reached", "changed", "success"]:
    data_def_pivot_latex.loc[avg_idx, col] = data_def_pivot.loc[avg_idx, col].apply(lambda x: f"{int(x)}/{N_IID}").values

# swap the column level
data_def_pivot_latex = data_def_pivot_latex.swaplevel(axis=1)
# sort the column level
data_def_pivot_latex = data_def_pivot_latex.sort_index(axis=1)
# change column order for to (BIC FIX) x (score reached changed success time)
data_def_pivot_latex = data_def_pivot_latex.reindex(
    columns=pd.MultiIndex.from_product(
        [["BIC", "FIX"], ["score", "reached", "changed", "success", "time"]]
    )
)

# show the columns
display(data_def_pivot_latex)


latex_str = data_def_pivot_latex.to_latex()
# remove [t] from multirow
latex_str = latex_str.replace("[t]", "").replace(
    r"\multicolumn{2}{c|}{Average} &  &",
    r"\multicolumn{2}{c|}{Average} &"
).replace(
    r"\cline{1-8}", r"\midrule"
).replace(
    r"\cline{1-12}", r"\midrule"
)



print(latex_str)
# print(data_def_pivot_latex.to_latex())

BIC          \
                                                             score reached   
\mujs                        \#65     \SLIDER{3.3em}{1.00}{circle}   10/10   
                             \#141    \SLIDER{3.3em}{0.67}{circle}   10/10   
                             \#145    \SLIDER{3.3em}{0.87}{circle}   10/10   
                             \#166    \SLIDER{3.3em}{0.37}{circle}   10/10   
\libxml                      \#535    \SLIDER{3.3em}{1.00}{circle}   10/10   
                             \#550    \SLIDER{3.3em}{0.40}{circle}   10/10   
                             \#553    \SLIDER{3.3em}{1.00}{circle}   10/10   
                             \#720    \SLIDER{3.3em}{0.33}{circle}   10/10   
                             \#841    \SLIDER{3.3em}{1.00}{circle}   10/10   
\poppler                     \#1282   \SLIDER{3.3em}{0.37}{circle}    8/10   
                             \#1289   \SLIDER{3.3em}{0.13}{circle}    4/10   
                             \#1303   \SLIDER{3.3em}{0.00}{circle}    0/10   
                             \#1305   \SLIDER{3.3em}{0.00}{circle}    0/10   
                             \#1381   \SLIDER{3.3em}{0.00}{circle}    0/10   
\jerryscript                 \#5013   \SLIDER{3.3em}{0.33}{circle}   10/10   
                             \#5117   \SLIDER{3.3em}{0.67}{circle}   10/10   
                             \#5138   \SLIDER{3.3em}{0.73}{circle}   10/10   
                             \#5153   \SLIDER{3.3em}{0.67}{circle}   10/10   
\zthree                      \#6659   \SLIDER{3.3em}{0.00}{circle}    0/10   
                             \#6914   \SLIDER{3.3em}{0.07}{circle}    2/10   
                             \#7246   \SLIDER{3.3em}{0.33}{circle}   10/10   
                             \#7252   \SLIDER{3.3em}{0.27}{circle}    8/10   
\php                         \#16777  \SLIDER{3.3em}{0.33}{circle}   10/10   
                             \#16978  \SLIDER{3.3em}{0.37}{circle}   10/10   
                             \#17442  \SLIDER{3.3em}{0.67}{circle}   10/10   
                             \#17463  \SLIDER{3.3em}{0.20}{circle}    6/10   
\jq                          \#2825   \SLIDER{3.3em}{0.33}{circle}   10/10   
                             \#2976   \SLIDER{3.3em}{0.33}{circle}   10/10   
                             \#3262   \SLIDER{3.3em}{1.00}{circle}   10/10   
                             \#49014  \SLIDER{3.3em}{0.13}{circle}    2/10   
\micropython                 \#13007  \SLIDER{3.3em}{0.53}{circle}    9/10   
                             \#13041  \SLIDER{3.3em}{0.00}{circle}    0/10   
                             \#17733  \SLIDER{3.3em}{0.33}{circle}   10/10   
                             \#17815  \SLIDER{3.3em}{0.63}{circle}   10/10   
                             \#17841  \SLIDER{3.3em}{0.67}{circle}   10/10   
                             \#17847  \SLIDER{3.3em}{0.67}{circle}   10/10   
\multicolumn{2}{c|}{Average}          \SLIDER{3.3em}{0.46}{circle}   31/36   

                                                             \
                                     changed success   time   
\mujs                        \#65      10/10   10/10    7.5   
                             \#141     10/10    0/10   45.4   
                             \#145      8/10    8/10   37.1   
                             \#166      1/10    0/10   57.6   
\libxml                      \#535     10/10   10/10    5.4   
                             \#550      2/10    0/10   27.6   
                             \#553     10/10   10/10    6.7   
                             \#720      0/10    0/10   53.6   
                             \#841     10/10   10/10    5.9   
\poppler                     \#1282     3/10    0/10  196.6   
                             \#1289     0/10    0/10  165.0   
                             \#1303     0/10    0/10  218.7   
                             \#1305     0/10    0/10  267.2   
                             \#1381     0/10    0/10  202.8   
\jerryscript

\begin{tabular}{llllllllllll}
\toprule
 &  & \multicolumn{5}{r}{BIC} & \multicolumn{5}{r}{FIX} \\
 &  & score & reached & changed & success & time & score & reached & changed & success & time \\
\midrule
\multirow{4}{*}{\mujs} & \#65 & \SLIDER{3.3em}{1.00}{circle} & 10/10 & 10/10 & 10/10 & 7.5 & \SLIDER{3.3em}{0.73}{circle} & 10/10 & 6/10 & 6/10 & 19.1 \\
 & \#141 & \SLIDER{3.3em}{0.67}{circle} & 10/10 & 10/10 & 0/10 & 45.4 & \SLIDER{3.3em}{0.70}{circle} & 10/10 & 10/10 & 1/10 & 31.7 \\
 & \#145 & \SLIDER{3.3em}{0.87}{circle} & 10/10 & 8/10 & 8/10 & 37.1 & \SLIDER{3.3em}{0.93}{circle} & 10/10 & 9/10 & 9/10 & 28.8 \\
 & \#166 & \SLIDER{3.3em}{0.37}{circle} & 10/10 & 1/10 & 0/10 & 57.6 & \SLIDER{3.3em}{1.00}{circle} & 10/10 & 10/10 & 10/10 & 7.4 \\
\midrule
\multirow{5}{*}{\libxml} & \#535 & \SLIDER{3.3em}{1.00}{circle} & 10/10 & 10/10 & 10/10 & 5.4 & \SLIDER{3.3em}{1.00}{circle} & 10/10 & 10/10 & 10/10 & 5.5 \\
 & \#550 & \SLIDER{3.3em}{0.40}{circle} & 10/10 & 2/10 & 0/10 & 27.6 & \SLID

### Table 3
Results for our ablation study with average effectiveness score on a slider.

In [7]:
config_prompt_msg = config_def.copy()
config_prompt_msg["git_info"] = "MSGONLY"
config_prompt_diff = config_def.copy()
config_prompt_diff["git_info"] = "DIFFONLY"
config_prompt_gen = config_def.copy()
config_prompt_gen["GENCMD"] = 1
config_llm_temp1 = config_def.copy()
config_llm_temp1["LLM_temp"] = 1
config_llm_mini = config_def.copy()
config_llm_mini["LLM"] = "gpt-4o-mini"
config_llm_deepseek = config_def.copy()
config_llm_deepseek["LLM"] = "deepseek-r1"
config_exec_iter10 = config_def.copy()
config_exec_iter10["max_iter"] = 10
config_exec_xfeedback = config_def.copy()
config_exec_xfeedback["NOFEEDBACK"] = 1

configs = {
    "default-": config_def,
    "prompt-msg": config_prompt_msg,
    "prompt-diff": config_prompt_diff,
    "prompt-gen": config_prompt_gen,
    "llm-temp1": config_llm_temp1,
    "llm-mini": config_llm_mini,
    "llm-deepseek": config_llm_deepseek,
    "exec-iter10": config_exec_iter10,
    "exec-xfeedback": config_exec_xfeedback,
}

data_each_config = []
for config_key, config in configs.items():
    data_config = data.copy()
    for key, value in config.items():
        if pd.isna(value):
            data_config = data_config[pd.isna(data_config[key])]
        else:
            data_config = data_config[data_config[key] == value]
    # check the number of datapoints for each config
    print(f"{config_key}: {len(data_config)}")
    # 66 commits in total, if len(data_config) != 660, print warning
    if len(data_config) != N_DATAPOINT:
        if config_key == "prompt-gen" and len(data_config) == 220:
            pass  # only 2+5+4 bugs for libxml2,poppler,jq need GENCMD, 11*10*2=220
        else:
            print(
                f"!!Warning!! {config_key} has {len(data_config)} data points; expected 460"
            )
    data_config = (
        data_config.groupby(["scenario", "subject", "iid"])
        .agg({"score": "mean"})
        .reset_index()
    )
    data_config["config_category"] = config_key.split("-")[0]
    data_config["config"] = config_key.split("-")[1]
    data_each_config.append(data_config)

data_each_config = pd.concat(data_each_config)
data_each_config = data_each_config.pivot(
    index=["scenario", "subject", "iid"],
    columns=["config_category", "config"],
    values="score",
)
# for the column ("prompt", "gen"), if the value is NaN, fill it with the value in ("default", "default")
data_each_config[("prompt", "gen")] = data_each_config[("prompt", "gen")].fillna(
    data_each_config[("default", "")]
)

# NOTE: FIX poppler 1305 (prompt, msg) data is wrong, if it is 0.2, manually change to 0.0
if data_each_config.loc[("FIX", "poppler", 1305), ("prompt", "msg")] == 0.2:
    data_each_config.loc[("FIX", "poppler", 1305), ("prompt", "msg")] = 0.0

# Unstack scenario from index to columns (BIC | FIX layout)
data_each_config = data_each_config.unstack(level=0)
# After unstack, columns are: (config_category, config, scenario)
# Reorder to: (scenario, config_category, config)
data_each_config.columns = data_each_config.columns.reorder_levels([2, 0, 1])

# Reorder columns to match configs dict order: default, prompt (msg, diff, gen), llm (temp1, mini, deepseek), exec (iter10, xfeedback)
scenario_order = ["BIC", "FIX"]
config_category_order = ["default", "prompt", "llm", "exec"]
config_order = {"default": [""], "prompt": ["msg", "diff", "gen"], "llm": ["temp1", "mini", "deepseek"], "exec": ["iter10", "xfeedback"]}

# Build the desired column order
desired_columns = []
for scenario in scenario_order:
    for category in config_category_order:
        for cfg in config_order[category]:
            desired_columns.append((scenario, category, cfg))

data_each_config = data_each_config[desired_columns]

# reorder the index level 0: ["mujs", "libxml2", "poppler", "jerryscript", "php-src", "z3", "jq", "micropython"]
data_each_config.index = pd.MultiIndex.from_arrays([
    pd.Categorical(data_each_config.index.get_level_values(0), categories=[
        "mujs", "libxml2", "poppler", "jerryscript", "z3", "php-src", "jq", "micropython"
    ], ordered=True),
    data_each_config.index.get_level_values(1)
])
data_each_config = data_each_config.sort_index()

# add average row
average = data_each_config.agg("mean").to_frame().T
average.index = pd.MultiIndex.from_tuples([("Average", "")])
data_each_config = pd.concat([data_each_config, average])

data_each_config

default-: 720
prompt-msg: 720
prompt-diff: 720
prompt-gen: 280
!!Warning!! prompt-gen has 280 data points; expected 460
llm-temp1: 720
llm-mini: 720
llm-deepseek: 720
exec-iter10: 720
exec-xfeedback: 720


scenario                BIC                                                    \
config_category     default    prompt                           llm             
config                            msg      diff       gen     temp1      mini   
mujs        65     3.000000  3.000000  3.000000  3.000000  3.000000  0.600000   
            141    2.000000  2.000000  2.000000  2.000000  2.000000  2.000000   
            145    2.600000  3.000000  2.600000  2.600000  2.600000  1.200000   
            166    1.100000  1.300000  1.000000  1.100000  1.200000  1.000000   
libxml2     535    3.000000  3.000000  3.000000  1.200000  3.000000  3.000000   
            550    1.200000  1.900000  1.100000  1.200000  1.500000  1.500000   
            553    3.000000  3.000000  3.000000  1.000000  3.000000  3.000000   
            720    1.000000  1.000000  1.000000  1.100000  1.000000  1.000000   
            841    3.000000  3.000000  3.000000  2.000000  3.000000  3.000000   
poppler     1282   1.100000  1.300000  1.000000  1.000000  1.400000  0.100000   
            1289   0.400000  0.600000  1.000000  0.900000  1.500000  0.400000   
            1303   0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
            1305   0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
            1381   0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
jerryscript 5013   1.000000  1.000000  1.000000  1.000000  1.100000  0.100000   
            5117   2.000000  2.000000  2.000000  2.000000  2.000000  2.000000   
            5138   2.200000  2.000000  2.300000  2.200000  2.200000  2.000000   
            5153   2.000000  2.000000  2.300000  2.000000  2.100000  2.000000   
z3          6659   0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
            6914   0.200000  0.100000  0.400000  0.200000  0.400000  0.200000   
            7246   1.000000  1.000000  1.000000  1.000000  1.000000  1.000000   
            7252   0.800000  0.000000  0.900000  0.800000  0.400000  0.000000   
php-src     16777  1.000000  0.600000  0.900000  1.000000  1.000000  0.900000   
            16978  1.100000  1.500000  1.300000  1.100000  1.300000  1.000000   
            17442  2.000000  1.700000  2.000000  2.000000  1.800000  1.800000   
            17463  0.600000  1.000000  0.300000  0.600000  0.700000  0.300000   
jq          2825   1.000000  1.200000  1.000000  1.000000  1.000000  1.000000   
            2976   1.000000  1.000000  1.000000  1.100000  1.000000  1.000000   
            3262   3.000000  3.000000  3.000000  1.100000  3.000000  0.000000   
            49014  0.400000  0.000000  0.400000  0.200000  0.600000  0.000000   
micropython 13007  1.600000  2.300000  1.100000  1.600000  1.500000  0.000000   
            13041  0.000000  0.300000  0.300000  0.000000  0.000000  0.000000   
            17733  1.000000  1.000000  1.000000  1.000000  1.000000  1.000000   
            17815  1.900000  1.800000  1.100000  1.900000  2.000000  1.900000   
            17841  2.000000  2.000000  2.000000  2.000000  2.300000  1.900000   
            17847  2.000000  2.000000  2.000000  2.000000  2.000000  0.000000   
Average            1.366667  1.405556  1.361111  1.191667  1.433333  0.969444   

scenario                                              FIX                  \
config_category                  exec             default    prompt         
config             deepseek    iter10 xfeedback                 msg  diff   
mujs        65     3.000000  3.000000  2.700000  2.200000  1.000000  2.20   
            141    2.000000  2.000000  2.000000  2.100000  0.000000  1.70   
            145    3.000000  3.000000  2.200000  2.800000  2.400000  2.70   
            166    1.800000  1.100000  1.100000  3.000000  3.000000  2.70   
libxml2     535    3.000000  3.000000  3.000000  3.000000  3.000000  3.00   
            550    2.000000  1.400000  1.100000  0.000000  0.000000  0.00   
            553    3.000000  3.000000  3.000000  3.000000  3.00000

In [8]:
# to latex
# Store raw values BEFORE any transformations
data_raw_values = data_each_config.copy()

data_each_config_latex = data_each_config.astype(object).copy()  # Convert to object dtype to allow strings

# index formats
data_each_config_latex.index = data_each_config_latex.index.set_levels(
    [
        data_each_config_latex.index.levels[0].map(
            {
                "mujs": r"\mujs",
                "libxml2": r"\libxml",
                "poppler": r"\poppler",
                "jerryscript": r"\jerryscript",
                "php-src": r"\php",
                "z3": r"\zthree",
                "jq": r"\jq",
                "micropython": r"\micropython",
                "Average": r"\multicolumn{2}{c||}{Average}",
            }
        ),
        data_each_config_latex.index.levels[1].map(
            lambda x: f"\\#{x}" if x else ""
        ),
    ]
)

###### column formats
# Format each cell: regular rows get slider, Average row gets slider + raw value below
# Need to use the original (unformatted) index for data_raw_values
for i, idx_formatted in enumerate(data_each_config_latex.index):
    idx_raw = data_raw_values.index[i]  # Get original index
    is_avg = idx_raw[0] == "Average"  # Check if this is the Average row
    for col in data_each_config_latex.columns:
        raw_value = data_raw_values.loc[idx_raw, col]
        if pd.notna(raw_value):
            scaled = raw_value / 3
            if is_avg:
                # Average row: nested tabular with slider and raw value
                data_each_config_latex.loc[idx_formatted, col] = f"\\begin{{tabular}}{{c}}\\SLIDER{{\\mysliderlen}}{{{scaled:.2f}}}{{circle}}\\vspace{{-0.1cm}}\\\\\\small{{{raw_value:.2f}}}\\end{{tabular}}"
            else:
                # Regular rows: just the slider
                data_each_config_latex.loc[idx_formatted, col] = f"\\SLIDER{{\\mysliderlen}}{{{scaled:.2f}}}{{circle}}"

display(data_each_config_latex[("BIC", "default")])

# Generate LaTeX table with proper formatting
latex_str = data_each_config_latex.to_latex()

print(latex_str)

/tmp/ipykernel_3608064/79301845.py:46: PerformanceWarning: indexing past lexsort depth may impact performance.
  display(data_each_config_latex[("BIC", "default")])


\mujs                          \#65                     \SLIDER{\mysliderlen}{1.00}{circle}
                               \#141                    \SLIDER{\mysliderlen}{0.67}{circle}
                               \#145                    \SLIDER{\mysliderlen}{0.87}{circle}
                               \#166                    \SLIDER{\mysliderlen}{0.37}{circle}
\libxml                        \#535                    \SLIDER{\mysliderlen}{1.00}{circle}
                               \#550                    \SLIDER{\mysliderlen}{0.40}{circle}
                               \#553                    \SLIDER{\mysliderlen}{1.00}{circle}
                               \#720                    \SLIDER{\mysliderlen}{0.33}{circle}
                               \#841                    \SLIDER{\mysliderlen}{1.00}{circle}
\poppler                       \#1282                   \SLIDER{\mysliderlen}{0.37}{circle}
                               \#1289                   \SLIDER{\mysliderlen}{0.

\begin{tabular}{llllllllllllllllllll}
\toprule
 & scenario & \multicolumn{9}{r}{BIC} & \multicolumn{9}{r}{FIX} \\
 & config_category & default & \multicolumn{3}{r}{prompt} & \multicolumn{3}{r}{llm} & \multicolumn{2}{r}{exec} & default & \multicolumn{3}{r}{prompt} & \multicolumn{3}{r}{llm} & \multicolumn{2}{r}{exec} \\
 & config &  & msg & diff & gen & temp1 & mini & deepseek & iter10 & xfeedback &  & msg & diff & gen & temp1 & mini & deepseek & iter10 & xfeedback \\
\midrule
\multirow[t]{4}{*}{\mujs} & \#65 & \SLIDER{\mysliderlen}{1.00}{circle} & \SLIDER{\mysliderlen}{1.00}{circle} & \SLIDER{\mysliderlen}{1.00}{circle} & \SLIDER{\mysliderlen}{1.00}{circle} & \SLIDER{\mysliderlen}{1.00}{circle} & \SLIDER{\mysliderlen}{0.20}{circle} & \SLIDER{\mysliderlen}{1.00}{circle} & \SLIDER{\mysliderlen}{1.00}{circle} & \SLIDER{\mysliderlen}{0.90}{circle} & \SLIDER{\mysliderlen}{0.73}{circle} & \SLIDER{\mysliderlen}{0.33}{circle} & \SLIDER{\mysliderlen}{0.73}{circle} & \SLIDER{\mysliderlen}{0.73}{c

In [ ]:
# subset the data regarding the config
data_def = data.copy()
for key, value in config_def.items():
    if pd.isna(value):
        data_def = data_def[pd.isna(data_def[key])]
    else:
        data_def = data_def[data_def[key] == value]

data_def_success = data_def[data_def['success'] == 1]
data_def_success_unique = data_def_success.drop_duplicates(subset=['subject', 'commit'])
display(data_def_success_unique)
data_def_success_unique = data_def_success_unique.groupby(['scenario']).size().reset_index(name='count')
display(data_def_success_unique)

data_r1 = data[data['LLM'] == 'deepseek-r1']
# count success of msgonly
data_r1_success = data_r1[data_r1['success'] == 1]
# unique by commit
data_r1_success_unique = data_r1_success.drop_duplicates(subset=['subject', 'commit'])
display(data_r1_success_unique)
# group by scenario
data_r1_success_unique = data_r1_success_unique.groupby(['scenario']).size().reset_index(name='count')
display(data_r1_success_unique)

Impact of iter 5 -> 10

In [ ]:
# config for default
config_def = {
    "git_info": "FULL",
    "max_iter": 5,
    "LLM": "gpt-4o-2024-08-06",
    "LLM_temp": 0.5,
    "NOFEEDBACK": np.nan,  # choose rows with NaN
    "GENCMD": np.nan,
    # "RUNFUZZ": np.nan,
}
# subset the data regarding the confiG
data_def = data.copy()
for key, value in config_def.items():
    if pd.isna(value):
        data_def = data_def[pd.isna(data_def[key])]
    else:
        data_def = data_def[data_def[key] == value]
# set index [scenario, subject, iid]
data_def = data_def.set_index(["scenario", "subject", "iid"])
# only FIX scenario
data_def = data_def.loc["FIX"]

config_def10 = {
    "git_info": "FULL",
    "max_iter": 10,
    "LLM": "gpt-4o-2024-08-06",
    "LLM_temp": 0.5,
    "NOFEEDBACK": np.nan,  # choose rows with NaN
    "GENCMD": np.nan,
    # "RUNFUZZ": np.nan,
}
# subset the data regarding the confiG
data_def10 = data.copy()
for key, value in config_def10.items():
    if pd.isna(value):
        data_def10 = data_def10[pd.isna(data_def10[key])]
    else:
        data_def10 = data_def10[data_def10[key] == value]
# set index [scenario, subject, iid]
data_def10 = data_def10.set_index(["scenario", "subject", "iid"])
# only FIX scenario
data_def10 = data_def10.loc["FIX"]

# merge the two dataframes
data_def = pd.merge(
    data_def,
    data_def10,
    on=["subject", "iid"],
    suffixes=("_5", "_10"),
)
# check the case where the score is different
# data_def_diff = data_def[
#     data_def["score_5"] != data_def["score_10"]
# ]
display(data_def[['score_5', 'score_10']])
# check the average for each (scenario, subject, iid)
data_def_diff_avg = data_def.groupby(["subject", "iid"]).agg(
    {"score_5": "mean", "score_10": "mean"}
)
display(data_def_diff_avg)
# count the number of 0, 1, 2, 3 in the score_5 and score_10 for each 
# (scenario, subject, iid)
data_def_diff_count = data_def.groupby(["subject", "iid"]).agg(
    {"score_5": "value_counts", "score_10": "value_counts"}
)
data_def_diff_count = data_def_diff_count.unstack()
# data_def_diff_count.columns = data_def_diff_count.columns.swaplevel(0, 1)
data_def_diff_count = data_def_diff_count.sort_index(axis=1)
data_def_diff_count

### (New RQ3): Impact of ENHANCED/REDUCED msg

In [ ]:
config_prompt_msg = config_def.copy()
config_prompt_msg["git_info"] = "MSGONLY"
config_prompt_diff = config_def.copy()
config_prompt_diff["git_info"] = "DIFFONLY"
config_prompt_enhanced = config_def.copy()
config_prompt_enhanced["git_info"] = "ENHANCED"
config_prompt_reduced = config_def.copy()
config_prompt_reduced["git_info"] = "REDUCED"


configs = {
    "default-": config_def,
    "prompt-msg": config_prompt_msg,
    "prompt-enhanced": config_prompt_enhanced,
    "prompt-reduced": config_prompt_reduced,
}

data_each_config = []
for config_key, config in configs.items():
    data_config = data.copy()
    for key, value in config.items():
        if pd.isna(value):
            data_config = data_config[pd.isna(data_config[key])]
        else:
            data_config = data_config[data_config[key] == value]
    data_config = data_config[data_config["scenario"] == "FIX"]
    # check the number of datapoints for each config
    print(f"{config_key}: {len(data_config)}")
    # 12 commits in total, if len(data_config) != 230, print warning
    if len(data_config) != 230:
        if config_key == "prompt-gen" and len(data_config) == 70:
            pass  # 4 commits in mujs doesn't need GENCMD
        else:
            print(
                f"!!Warning!! {config_key} has {len(data_config)} data points; expected 230"
            )
    data_config = (
        data_config.groupby(["scenario", "subject", "iid"])
        .agg({"score": "mean"})
        .reset_index()
    )
    data_config["config_category"] = config_key.split("-")[0]
    data_config["config"] = config_key.split("-")[1]
    data_each_config.append(data_config)

data_each_config = pd.concat(data_each_config)
data_each_config = data_each_config.pivot(
    index=["scenario", "subject", "iid"],
    columns=["config_category", "config"],
    values="score",
)

dfs = []
# add average row for each scenario
for scenario in data_each_config.index.levels[0]:
    print(f"Scenario: {scenario}")
    # scenario is the first level of the index
    sub_data = data_each_config.loc[
        data_each_config.index.get_level_values(0) == scenario
    ]
    average = sub_data.agg("mean").to_frame().T
    average.index = pd.MultiIndex.from_tuples([(scenario, "Average", "")])
    # data_each_config = pd.concat([data_each_config, average])
    dfs = dfs + [sub_data, average]
# data_each_config = data_each_config.sort_index()
data_each_config = pd.concat(dfs)
# reorder the index level 1: ["mujs", "libxml2", "poppler", "jerryscript", "php-src", "z3", "Average"]
data_each_config.index = pd.MultiIndex.from_arrays([
    pd.Categorical(data_each_config.index.get_level_values(0), categories=[
        "FIX"  # No BIC for enhanced/reduced
    ], ordered=True),
    pd.Categorical(data_each_config.index.get_level_values(1), categories=[
        "mujs", "libxml2", "poppler", "jerryscript", "z3", "php-src", "jq", "micropython", "Average"
    ], ordered=True),
    data_each_config.index.get_level_values(2)
])
data_each_config = data_each_config.sort_index()
data_each_config

In [ ]:
# to latex
data_each_config_latex = data_each_config.copy()
# index formats
data_each_config_latex.index = data_each_config_latex.index.set_levels(
    [
        data_each_config_latex.index.levels[0],
        data_each_config_latex.index.levels[1].map(
            {
                "mujs": r"\mujs",
                "libxml2": r"\libxml",
                "poppler": r"\poppler",
                "jerryscript": r"\jerryscript",
                "php-src": r"\php",
                "z3": r"\zthree",
                "jq": r"\jq",
                "micropython": r"\micropython",
                "Average": r"\multicolumn{2}{c|}{Average}",
            }
        ),
        data_each_config_latex.index.levels[2].map(
            lambda x: f"\\#{x}" if x else ""
        ),
    ]
)
###### column formats
# score -> \SLIDER{3.3em}{score / 3:.2f}{circle}
data_each_config_latex = data_each_config_latex.applymap(
    lambda x: f"\\SLIDER{{3.3em}}{{{(x/3):.2f}}}{{circle}}"
)
display(data_each_config_latex)
print(data_each_config_latex.to_latex().replace("[t]", "").replace(
    r"\multirow{24}{*}{BIC}", r"\multirow{24}{*}{\begin{sideways}Bug-finding\end{sideways}}"
).replace(
    r"\multirow{24}{*}{FIX}", r"\multirow{24}{*}{\begin{sideways}Bug-reproduction\end{sideways}}"
).replace(
    r"\multicolumn{2}{c|}{Average} &  &",
    r"\multicolumn{2}{c|}{Average} &"
).replace(
    r"\cline{1-11} \cline{2-11}", r"\midrule"
).replace(
    r"\cline{2-11}", r"\cmidrule{2-11}"
))

# Table 4

## WAFLGo results

In [ ]:
import glob

# init_result = [
#     ["BIC", "mujs", 65, "F"],
#     ["BIC", "mujs", 141, "F"],
#     ["BIC", "mujs", 145, "F"],
#     ["BIC", "mujs", 166, "B"],
#     ["BIC", "libxml2", 535, "B"],
#     ["BIC", "libxml2", 550, "R"],
#     ["BIC", "poppler", 1282, "R"],
#     ["BIC", "poppler", 1289, "R"],
#     ["BIC", "poppler", 1303, "F"],
#     ["BIC", "poppler", 1305, "F"],
#     ["BIC", "poppler", 1381, "R"],
#     ["FIX", "mujs", 65, "R"],
#     ["FIX", "mujs", 141, "F"],
#     ["FIX", "mujs", 145, "F"],
#     ["FIX", "mujs", 166, "B"],
#     ["FIX", "libxml2", 535, "B"],
#     ["FIX", "libxml2", 550, "F"],
#     ["FIX", "poppler", 1282, "R"],
#     ["FIX", "poppler", 1289, "R"],
#     ["FIX", "poppler", 1303, "F"],
#     ["FIX", "poppler", 1305, "R"],
#     ["FIX", "poppler", 1381, "R"],
# ]


def parse_init(data_path):
    data = pd.read_csv(data_path, sep=r"\s\|\s", engine="python")
    data["scenario"] = data_path.split("_")[-2]
    data["subject"] = data_path.split("_")[-3]
    data["init"] = data["final"].map(
        {"N": "F", "X": "F", "B": "B", "D": "D", "R": "R"}
    )
    data["iid"] = data["issue"]
    return data[["scenario", "subject", "iid", "init"]]


# parse_init("/home/nfs0/smbshares/softsec/seedscheck_mujs_BIC_withiid.csv")
init_result = pd.concat(
    [parse_init(f) for f in glob.glob("fuzz/seedscheck_*_withiid.csv")]
)
init_result.set_index(["scenario", "subject", "iid"], inplace=True)
init_result.sort_index(inplace=True)
init_result

In [ ]:
# parse WAFLGo result

# index = ["scenario", "iid"]
# columns = [["WAFLGo", "WAFLGo"], ["Result", "Time"]]

# Function to parse each waflgo_*.csv file
import glob


def parse_fuzz_csv(file_path):
    try:
        df = pd.read_csv(file_path, sep=r"\t", engine="python")
    except Exception as e:  # use different separator
        df = pd.read_csv(file_path, sep=r"\s\|\s", engine="python")
    print(f"Reading {file_path} with shape {df.shape}")
    df.columns = df.columns.str.strip()
    if 'scenario' not in df.columns:
        df["scenario"] = file_path.split(".")[0].split("_")[
            -2
        ]  # Extract scenario from file name
    if 'subject' not in df.columns:
        df["subject"] = file_path.split(".")[0].split("_")[
            1
        ]  # Extract subject from file name
    df["iid"] = df["issue"]
    df["success"] = df["final"].apply(
        lambda x: 1 if x == "B" else 0
    )  # Success if final is 'B'
    try:
        df["time"] = (
            df["time to find first crash in ms"].apply(
                pd.to_numeric, errors="coerce"
            )
            / 1000
        )  # Convert time to seconds
    except KeyError:  # postfuzz_*.csv
        df["time"] = (
            df["time to find first crash"].apply(pd.to_numeric, errors="coerce")
            / 1000
        )  # Convert time to seconds
    df.loc[df["success"] == 0, "time"] = (
        np.nan
    )  # some rows have time but not success, should be NaN
    # only keep columns: scenario, subject, iid, commit, idx, success, time
    df = df[["scenario", "subject", "iid", "commit", "idx", "success", "time"]]
    return df


# Parse and concatenate all CSV files
# csv_files = glob.glob("fuzz/waflgo_*_withiid.csv")
# waflgo_data = pd.concat([parse_fuzz_csv(file) for file in csv_files])
waflgo_data = parse_fuzz_csv("fuzz/merged_waflgo.csv")
display(waflgo_data.head())
# Group by scenario and iid, and aggregate success and time
waflgo_result = (
    waflgo_data.groupby(["scenario", "subject", "iid"])
    .agg(
        {
            "success": "sum",  # Number of successes
            "time": "mean",  # Average time in seconds
        }
    )
    .reset_index()
)

# Set index as scenario and iid
waflgo_result.set_index(["scenario", "subject", "iid"], inplace=True)

# Merge with initial results
waflgo_result = waflgo_result.join(init_result, how="outer")
# Reorder columns
waflgo_result = waflgo_result[["init", "success", "time"]]
# for the cases there init is B, set success 10
waflgo_result.loc[waflgo_result["init"] == "B", "success"] = 10
# for the cases there success is not NaN but time is NaN, set time 24 hours
waflgo_result.loc[
    waflgo_result["success"].notna() & waflgo_result["time"].isna(), "time"
] = 24 * 3600
# rename columns
waflgo_result.columns = pd.MultiIndex.from_product([["WAFLGo"], ["Init", "Bug", "Time"]])
waflgo_result

## Cleverest results

In [ ]:
# config for default
config_def = {
    "git_info": "FULL",
    "max_iter": 5,
    "LLM": "gpt-4o-2024-08-06",
    "LLM_temp": 0.5,
    "NOFEEDBACK": np.nan,  # choose rows with NaN
    "GENCMD": np.nan,
    # "RUNFUZZ": np.nan,
}
# subset the data regarding the confiG
data_def = data.copy()
for key, value in config_def.items():
    if pd.isna(value):
        data_def = data_def[pd.isna(data_def[key])]
    else:
        data_def = data_def[data_def[key] == value]
# check the number of datapoints; expected to 460 = 23 iid * 2 scenario * 10 repetition; if not, print warning
if len(data_def) != 460:
    print(
        f"!!Warning!!: the number of datapoints is {len(data_def)}; expected to be 460"
    )
# aggregate the data regarding the scenario and iid
data_def["reached"] = data_def["score"].apply(lambda x: 1 if x >= 1 else 0)
data_def["changed"] = data_def["score"].apply(lambda x: 1 if x >= 2 else 0)

data_def = (
    data_def.groupby(["scenario", "subject", "iid"])
    # .agg({"score": "mean", "success": "sum", "time": "mean"})
    .agg({"score": "mean", "reached": "sum", "changed": "sum", "success": "sum", "time": "mean"})
    .reset_index()
)

display(data_def.tail())
data_def_tb4 = data_def.copy()
data_def_tb4 = data_def_tb4.groupby(["scenario", "subject", "iid"]).agg(
    # {"score": "mean", "success": "sum", "time": "mean"}
    {"score": "mean", "reached": "sum", "changed": "sum", "success": "sum", "time": "mean"}
)
# rename columns
data_def_tb4.columns = pd.MultiIndex.from_product([["Cleverest"], ["Score", "Reach", "Change", "Bug", "Time"]])
data_def_tb4

## ClevFuzz results

In [ ]:
data_def[(data_def["subject"] == "poppler") & (data_def["scenario"] == "BIC") & (data_def["iid"] == 1282)]

In [ ]:
data_def = data.copy()
data_def

In [ ]:
data_def[(data_def["file"].str.contains("-1127"))]

In [ ]:
# Read all postfuzz_*.csv files
# csv_files = glob.glob("fuzz/postfuzz_*_withiid.csv")
# print(csv_files)
# csv_files = [f for f in csv_files if "withiid" not in f]
# fuzz_data = pd.concat([parse_fuzz_csv(file) for file in csv_files])
fuzz_data = parse_fuzz_csv("fuzz/merged_postfuzz.csv")
display(fuzz_data)

# for each row, find row from data_def with the 'file' includes the 'idx' in
# fuzz_data and 'iid' and 'commit' are the same; if not found, print error
clevfuzz_data = []
for idx, row in fuzz_data.iterrows():
    corr_row = data_def[
        (data_def["file"].str.contains(row["idx"]))
        & (data_def["iid"] == row["iid"])
        & (data_def["commit"] == row["commit"])
    ]
    if len(corr_row) != 1:
        print(f"Error: {len(corr_row)} rows found for {row['idx']}")
        # display full rows
        pd.set_option("display.max_columns", None)
        pd.set_option("display.width", None)
        pd.set_option("display.max_colwidth", None)
        display(corr_row)
        break
    # check if final result was either "R" or "B", if so, add to clevfuzz_data
    try:
        if (corr_row["final_result"].values[0] in ["R", "D"]) or (
            (
                corr_row[["scenario", "subject"]].values[0] == ("BIC", "mujs")
            ).all()
            and (corr_row["iid"].values[0] == 141)
        ):
            new_row = row.copy()
            new_row["cleverest"] = corr_row["final_result"].values[0]
            clevfuzz_data.append(new_row)
    except KeyError:
        display(corr_row)
        break
pd.reset_option("display.max_columns")
pd.reset_option("display.width")
pd.reset_option("display.max_colwidth")
clevfuzz_data = pd.DataFrame(clevfuzz_data)
display(clevfuzz_data)
clevfuzz_data.set_index(["scenario", "subject", "iid"], inplace=True)
clevfuzz_data.sort_index(inplace=True)
display(clevfuzz_data.loc[("BIC", "mujs", 141), :])
display(clevfuzz_data.loc[("BIC", "poppler", 1282), :])
display(clevfuzz_data.loc[("BIC", "mujs", 166), :])
display(clevfuzz_data.loc[("FIX", "mujs", 65), :])

# set 24 hours to the cases where time is NaN
clevfuzz_data.loc[clevfuzz_data["time"].isna(), "time"] = 24 * 60 * 60
# add row 'try' to count the number of tries
clevfuzz_data["try"] = 1
clevfuzz_data

In [ ]:
clevfuzz_result = clevfuzz_data.groupby(["scenario", "subject", "iid"]).agg(
    {"try": "sum", "success": "sum", "time": "mean"}
)
# bug: "{success}/{try}"
clevfuzz_result["bug"] = (
    clevfuzz_result["success"].astype(str)
    + "/"
    + clevfuzz_result["try"].astype(str)
)
clevfuzz_result = clevfuzz_result[["bug", "time", "success"]]
# rename columns
clevfuzz_result.columns = pd.MultiIndex.from_product(
    [["ClevFuzz"], ["Bug", "Time", "Success"]]
)
clevfuzz_result

In [ ]:
# merge waflgo_result, data_def_tb4, and clevfuzz_result
tb4 = waflgo_result.join(data_def_tb4, how="outer")
tb4 = tb4.join(clevfuzz_result, how="outer")
tb4.loc[:, ("ClevFuzz", "BugAll")] = tb4.apply(
    lambda x: (
        x["Cleverest"]["Bug"]
        if pd.isna(x["ClevFuzz"]["Bug"])
        else x["ClevFuzz"]["Success"] + x["Cleverest"]["Bug"]
    ),
    axis=1,
)
# drop ["ClevFuzz", "Success"]
tb4 = tb4.drop(("ClevFuzz", "Success"), axis=1)

# sort the index: ["Scenario", "Subject", "Issue"]
tb4.index = pd.MultiIndex.from_arrays(
    [
        pd.Categorical(
            tb4.index.get_level_values(0),
            categories=["BIC", "FIX"],
            ordered=True,
        ),
        pd.Categorical(
            tb4.index.get_level_values(1),
            categories=[
                "mujs",
                "libxml2",
                "poppler",
                "jerryscript",
                "z3",
                "php-src",
                "Aggregate",
            ],
            ordered=True,
        ),
        tb4.index.get_level_values(2),
    ],
    names=["Scenario", "Subject", "Issue"],
)
tb4 = tb4.sort_index()
tb4 = tb4[tb4["WAFLGo"]["Time"].notna()]  # keep only rows supported by WAFLGo
tb4

In [ ]:
# create the aggregation row
agg_rows = []
for scenario in tb4.index.levels[0]:
    print(f"Scenario: {scenario}")
    sub_data = tb4.loc[tb4.index.get_level_values(0) == scenario]
    len_sub_data = len(sub_data)
    # waflgo_bug aggregation: count the number of non-NaN or non-zero values
    waflgo_bug = (sub_data["WAFLGo"]["Bug"] > 0).sum()
    # waflgo_time aggregation: mean of non-NaN values
    col = sub_data["WAFLGo"]["Time"]
    col_nonnan = col[~col.isna()]
    waflgo_time = col_nonnan.mean()
    # cleverest_score aggregation: mean of values
    cleverest_score = sub_data["Cleverest"]["Score"].mean()
    # cleverest_bug aggregation: count the number of non-zero values
    cleverest_bug = (sub_data["Cleverest"]["Bug"] > 0).sum()
    cleverest_change = (sub_data["Cleverest"]["Change"] > 0).sum()
    cleverest_reach = (sub_data["Cleverest"]["Reach"] > 0).sum()
    # cleverest_time aggregation: mean of values
    cleverest_time = sub_data["Cleverest"]["Time"].mean()
    # clevfuzz_bugall aggregation: count the number of non-zero values
    clevfuzz_bugall = (sub_data["ClevFuzz"]["BugAll"] > 0).sum()
    agg_rows.append(
        {
            # ("Scenario", "Subject", "Issue"): (scenario, "Aggregate", ""),
            "Scenario": scenario,
            "Subject": "Aggregate",
            "Issue": "",
            ("WAFLGo", "Init", ""): "",
            ("WAFLGo", "Bug", ""): f"{waflgo_bug}/{len_sub_data}",
            ("WAFLGo", "Time", ""): str(
                pd.to_datetime(waflgo_time, unit="s").strftime("%H:%M:%S")
            ),
            (
                "Cleverest",
                "Score",
                "",
            ): f"\\SLIDER{{3.3em}}{{{(cleverest_score/3):.2f}}}{{circle}}",
            ("Cleverest", "Reach", ""): f"{cleverest_reach}/{len_sub_data}",
            ("Cleverest", "Change", ""): f"{cleverest_change}/{len_sub_data}",
            ("Cleverest", "Bug", ""): f"{cleverest_bug}/{len_sub_data}",
            ("Cleverest", "Time", ""): str(
                pd.to_datetime(cleverest_time, unit="s").strftime("%H:%M:%S")
            ),
            ("ClevFuzz", "Bug", ""): "",
            ("ClevFuzz", "Time", ""): "",
            ("ClevFuzz", "BugAll", ""): f"{clevfuzz_bugall}/{len_sub_data}",
        }
    )
agg_rows = pd.DataFrame(agg_rows)
# to multiindex
agg_rows.set_index(["Scenario", "Subject", "Issue"], inplace=True)
# display(agg_rows)
agg_rows.columns = pd.MultiIndex.from_tuples(agg_rows.columns)
display(agg_rows)

In [ ]:
# format
tb4_latex = tb4.copy()
## index formats
tb4_latex.index = tb4_latex.index.set_levels(
    [
        tb4_latex.index.levels[0],
        tb4_latex.index.levels[1].map(
            {
                "mujs": r"\mujs",
                "libxml2": r"\libxml",
                "poppler": r"\poppler",
                "jerryscript": r"\jerryscript",
                "z3": r"\zthree",
                "jq": r"\jq",
                "micropython": r"\micropython",
                "php-src": r"\php",
                "Aggregate": "Aggregate",
            }
        ),
        tb4_latex.index.levels[2].map(lambda x: f"\\#{x}" if x else ""),
    ],
)
tb4_latex.index = pd.MultiIndex.from_arrays(
    [
        pd.Categorical(
            tb4_latex.index.get_level_values(0),
            categories=["BIC", "FIX"],
            ordered=True,
        ),
        pd.Categorical(
            tb4_latex.index.get_level_values(1),
            categories=[
                r"\mujs",
                r"\libxml",
                r"\poppler",
                r"\jerryscript",
                r"\zthree",
                r"\php",
                "Aggregate",
            ],
            ordered=True,
        ),
        tb4_latex.index.get_level_values(2),
    ],
    names=["Scenario", "Subject", "Issue"],
)
## column formats
### ["WAFLGo", "Init"]:
###  "F" -> r"\xmark",
###  "B" -> r"\cmark",
###  "R" -> r"\rmark",
###  NaN -> "-"
tb4_latex[("WAFLGo", "Init")] = tb4_latex[("WAFLGo", "Init")].map(
    {"F": r"\xmark", "B": r"\cmark", "R": r"\rmark", np.nan: "-"}
)
### ["WAFLGo", "Bug"]:
###  if number, "{int(x)}/10"
###  if NaN, "-"
tb4_latex[("WAFLGo", "Bug")] = tb4_latex[("WAFLGo", "Bug")].map(
    lambda x: f"{int(x)}/10" if not pd.isna(x) else "-"
)
### ["WAFLGo", "Time"]:
###  if NaN, "-"
###  if 86400, "T.O."
###  else, hh:mm:ss
tb4_latex[("WAFLGo", "Time")] = tb4_latex[("WAFLGo", "Time")].map(
    lambda x: (
        "-"
        if pd.isna(x)
        else (
            "T.O."
            if x == 86400
            else str(pd.to_datetime(x, unit="s").strftime("%H:%M:%S"))
        )
    )
)
### ["Cleverest", "Score"]:
###  f"\\SLIDER{{3.3em}}{{{(x/3):.2f}}}{{circle}}"
tb4_latex[("Cleverest", "Score")] = tb4_latex[("Cleverest", "Score")].map(
    lambda x: f"\\SLIDER{{3.3em}}{{{(x/3):.2f}}}{{circle}}"
)
### ["Cleverest", "Bug"]:
###  "{int(x)}/10"
tb4_latex[("Cleverest", "Reach")] = tb4_latex[("Cleverest", "Reach")].map(
    lambda x: f"{int(x)}/10"
)
tb4_latex[("Cleverest", "Change")] = tb4_latex[("Cleverest", "Change")].map(
    lambda x: f"{int(x)}/10"
)
tb4_latex[("Cleverest", "Bug")] = tb4_latex[("Cleverest", "Bug")].map(
    lambda x: f"{int(x)}/10"
)
### ["Cleverest", "Time"]:
###  hh:mm:ss
tb4_latex[("Cleverest", "Time")] = tb4_latex[("Cleverest", "Time")].map(
    lambda x: str(pd.to_datetime(x, unit="s").strftime("%H:%M:%S"))
)
### ["ClevFuzz", "Bug"]:
###  if NaN, "-/-"
###  else, as it is
tb4_latex[("ClevFuzz", "Bug")] = tb4_latex[("ClevFuzz", "Bug")].map(
    lambda x: x if not pd.isna(x) else "-/-"
)
### ["ClevFuzz", "Time"]:
###  if NaN, "-"
###  if 86400, "T.O."
###  else, hh:mm:ss
tb4_latex[("ClevFuzz", "Time")] = tb4_latex[("ClevFuzz", "Time")].map(
    lambda x: (
        "-"
        if pd.isna(x)
        else (
            "T.O."
            if x == 86400
            else str(pd.to_datetime(x, unit="s").strftime("%H:%M:%S"))
        )
    )
)
### ["ClevFuzz", "BugAll"]:
### "{int(x)}/10"
tb4_latex[("ClevFuzz", "BugAll")] = tb4_latex[("ClevFuzz", "BugAll")].map(
    lambda x: f"{int(x)}/10"
)

# Add the aggregation row
tb4_latex.loc[("BIC", r"\multicolumn{2}{c|}{Aggregate}", ""), :] = tuple(
    agg_rows.loc[("BIC", "Aggregate", "")]
)
tb4_latex.loc[("FIX", r"\multicolumn{2}{c|}{Aggregate}", ""), :] = tuple(
    agg_rows.loc[("FIX", "Aggregate", "")]
)

# sort the index
tb4_latex.index = pd.MultiIndex.from_arrays(
    [
        pd.Categorical(
            tb4_latex.index.get_level_values(0),
            categories=["BIC", "FIX"],
            ordered=True,
        ),
        pd.Categorical(
            tb4_latex.index.get_level_values(1),
            categories=[
                r"\mujs",
                r"\libxml",
                r"\poppler",
                r"\jerryscript",
                r"\zthree",
                r"\php",
                r"\multicolumn{2}{c|}{Aggregate}",
            ],
            ordered=True,
        ),
        pd.Categorical(
            tb4_latex.index.get_level_values(2),
            categories=tb4_latex.index.get_level_values(2).unique(),
            ordered=True,
        ),
    ],
    names=["Scenario", "Subject", "Issue"],
)
tb4_latex = tb4_latex.sort_index()

display(tb4_latex)
tb4_latex = (
    tb4_latex.to_latex()
    .replace("[t]", "")
    .replace(
        r"\multirow{24}{*}{BIC}",
        r"\multirow{24}{*}{\begin{sideways}Bug-finding\end{sideways}}",
    )
    .replace(
        r"\multirow{24}{*}{FIX}",
        r"\multirow{24}{*}{\begin{sideways}Bug-reproduction\end{sideways}}",
    )
    .replace(
        r"\multirow{15}{*}{BIC}",
        r"\multirow{15}{*}{\begin{sideways}Bug-finding\end{sideways}}",
    )
    .replace(
        r"\multirow{15}{*}{FIX}",
        r"\multirow{15}{*}{\begin{sideways}Bug-reproduction\end{sideways}}",
    )
    .replace(
        r"\multicolumn{2}{c|}{Average} &  &", r"\multicolumn{2}{c|}{Average} &"
    )
    # .replace(r"\cline{1-12} \cline{2-12}", r"\midrule")
    # .replace(r"\cline{2-12}", r"\cmidrule{2-12}")
    .replace(r"\cline{1-14} \cline{2-14}", r"\midrule")
    .replace(r"\cline{2-14}", r"\cmidrule{2-14}")
    .replace(
        r"\multicolumn{2}{c|}{Aggregate} &  &",
        r"\multicolumn{2}{c|}{Aggregate} &",
    )
)
print(tb4_latex)

# Figure 3

In [ ]:
clevfuzz_data[["cleverest", "success"]]

data = {}
for scenario in ["BIC", "FIX"]:
    sub_data = clevfuzz_data.loc[scenario]
    # count the number of R in cleverest and 1 in success
    R_B = len(
        sub_data[
            (sub_data["cleverest"] == "R") & (sub_data["success"] == 1)
        ]
    )
    D_B = len(
        sub_data[
            (sub_data["cleverest"] == "D") & (sub_data["success"] == 1)
        ]
    )
    R_F = len(
        sub_data[
            (sub_data["cleverest"] == "R") & (sub_data["success"] == 0)
        ]
    )
    D_F = len(
        sub_data[
            (sub_data["cleverest"] == "D") & (sub_data["success"] == 0)
        ]
    )
    data[scenario] = {
        "R_B": R_B,
        "D_B": D_B,
        "R_F": R_F,
        "D_F": D_F,
    }
clevfuzz_convert = pd.DataFrame(data).T
clevfuzz_convert["B"] = clevfuzz_convert["R_B"] + clevfuzz_convert["D_B"]
clevfuzz_convert["F"] = clevfuzz_convert["R_F"] + clevfuzz_convert["D_F"]
# column order = ["R_B", "D_B", "B", "R_F", "D_F", "F"]
clevfuzz_convert = clevfuzz_convert[
    ["R_B", "D_B", "B", "R_F", "D_F", "F"]
]
clevfuzz_convert.loc["sum"] = clevfuzz_convert.sum()
clevfuzz_convert

In [ ]:
tb4_time = tb4.copy()
## only rows where WAFLGo time is not NaN
tb4_time = tb4_time[tb4_time["WAFLGo"]["Time"].notna()]
## total time = cleverest time if clevfuzz time is NaN, else sum of cleverest
## and clevfuzz time
tb4_time["ClevFuzz", "TotalTime"] = tb4_time["Cleverest"]["Time"]
tb4_time.loc[
    tb4_time["ClevFuzz"]["Time"].notna(), ("ClevFuzz", "TotalTime")
] = (tb4_time["Cleverest"]["Time"] + tb4_time["ClevFuzz"]["Time"])
display(tb4_time)
# average time for each scenario
tb4_time_avg = tb4_time.groupby("Scenario").agg(
    {("WAFLGo", "Time"): "mean", ("ClevFuzz", "TotalTime"): "mean"}
)
# format to hh:mm:ss
tb4_time_avg = tb4_time_avg.applymap(
    lambda x: str(pd.to_datetime(x, unit="s").strftime("%H:%M:%S"))
)
tb4_time_avg

# Comment statistics

In [ ]:
# read columns msgwords,msgchars,difflines,diffchars from stats_22commits.csv, insert into data_each_config
# there are already "scenario" and "iid" columns in the stats_22commits.csv, so we can merge them
stats_path = "../stat_22commits.csv"
stats = pd.read_csv(stats_path)
# flatten multi-level column index
data_stats = data_each_config.copy()
# data_stats = data_stats.reset_index()
data_stats.columns = data_stats.columns.get_level_values(1)
data_stats = data_stats.reset_index()
# rename data_stats[2] column to data_stats['default]
data_stats = data_stats.rename(columns={data_stats.columns[2]: "default"})
print(data_stats.columns)
data_stats = data_stats[["scenario", "iid", "default", "msg", "diff"]]
data_stats_merged = data_stats.merge(stats, on=["scenario", "iid"])
# # set index back
data_stats_merged
# print as csv
data_stats_merged.to_csv("data_stats_merged.csv", index=False)